# B10 — vec2vec Demo: Translation Without Pairs

**The claim being demonstrated** (Platonic Representation Hypothesis /
vec2vec): embedding spaces of independently trained models share their
*shape* so strongly that translation between them is recoverable from
geometry alone — **no paired examples at all**.

**The setup that makes this honest:** the project's 4,000 image pairs are
split into two *disjoint image sets*:

- Side X: MobileCLIP embeddings of images 0-1999
- Side Y: SigLIP embeddings of images 2000-3999

No image appears on both sides, so nothing links the sets except the
shape of the spaces. A translator is learned unsupervised, then **graded
on the held-out true pairs** the algorithm never saw.

**Method** (a simplified, transparent cousin of vec2vec; the same
family as Alvarez-Melis & Jaakkola 2018 and MUSE):
1. **Gromov-Wasserstein optimal transport**: align the two *intra-space
   distance matrices* directly — the mathematical formalization of
   "matching shapes without correspondences" — yielding initial
   pseudo-pairs from geometry alone
2. **CSLS self-learning loop**: ridge-fit W on pseudo-pairs -> map X
   into Y's space -> extract *mutual* nearest neighbors under CSLS
   (hubness-corrected similarity) as better pseudo-pairs -> refit ->
   iterate

The algorithm was validated on a synthetic worst-case testbed (40
statistically interchangeable clusters): R@1 = 144x chance, 29% of the
supervised reference — passing the criteria below before touching real
data.

**Grading** on 500 true held-out pairs: cosine-to-target and retrieval
R@1/5/10, against three references — supervised ridge (upper), random
matrix (floor), chance = 1/500 = 0.2%.

**Pre-registered criteria:** PASS if unsupervised R@1 >= 10x chance AND
>= 25% of the supervised adapter; STRONG if >= 50% of supervised.
Runtime: ~1-2 min after loading. Needs `pairs.npz`.

In [ ]:
# 1. Storage
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DATA_DIR'] = '/content/drive/MyDrive/convergence_experiment'
print('DATA_DIR =', os.environ['DATA_DIR'])

In [ ]:
# 2. Build the unpaired setting from the project's data
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])
rng = np.random.default_rng(0)

pairs = np.load(DATA_DIR / 'pairs.npz')
mob, sig = pairs['mob_img'], pairs['sig_img']     # both L2-normalized

X_tr = mob[:2000]        # MobileCLIP side: images 0-1999
Y_tr = sig[2000:4000]    # SigLIP side: images 2000-3999 (DISJOINT)
# held-out TRUE pairs for grading only (never used in training):
X_te, Y_te = mob[3500:4000], sig[3500:4000]
# note: X_te images overlap Y_tr's set but the PAIRING is never revealed;
# to keep grading fully clean we exclude them from Y_tr:
Y_tr = sig[2000:3500]
print(f'unpaired training: X {X_tr.shape} (imgs 0-1999), '
      f'Y {Y_tr.shape} (imgs 2000-3499)')
print(f'grading pairs: {X_te.shape[0]} true (mob, sig) pairs, '
      f'chance R@1 = {1/len(X_te):.3%}')

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-9)

def ridge(X, Y, a=1e-2):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + a * np.eye(d), X.T @ Y)

def grade(W, label, refs=None):
    P = l2n(X_te @ W)
    cos = float((P * Y_te).sum(1).mean())
    sims = P @ Y_te.T
    ranks = (-sims).argsort(1)
    n = len(X_te)
    r = {k: float((ranks[:, :k] == np.arange(n)[:, None]).any(1).mean())
         for k in (1, 5, 10)}
    line = (f"{label:<34} cos={cos:.3f}  R@1={r[1]:.3f}  "
            f"R@5={r[5]:.3f}  R@10={r[10]:.3f}")
    if refs:
        line += f"   ({100*r[1]/max(refs[1],1e-9):.0f}% of supervised R@1)"
    print(line)
    return r

In [ ]:
# 3. References: supervised ceiling and random floor
W_sup = ridge(mob[:1500], sig[:1500])      # supervised, disjoint from test
r_sup = grade(W_sup, 'SUPERVISED ridge (reference)')
W_rand = rng.standard_normal((512, 768)) * 0.03
r_rand = grade(W_rand, 'random matrix (floor)')

In [ ]:
!pip -q install pot

In [ ]:
# 4. Step 1: Gromov-Wasserstein initialization - matching the shapes
import ot, warnings
warnings.filterwarnings('ignore')
N_GW, EPS = 800, 3e-3

ix = rng.choice(len(X_tr), N_GW, replace=False)
iy = rng.choice(len(Y_tr), N_GW, replace=False)
Cx = 1 - X_tr[ix] @ X_tr[ix].T          # intra-space cosine distances
Cy = 1 - Y_tr[iy] @ Y_tr[iy].T          # (the two 'shapes')
p = np.ones(N_GW) / N_GW
print('solving entropic Gromov-Wasserstein on the two distance '
      'matrices...')
T = ot.gromov.entropic_gromov_wasserstein(
    Cx, Cy, p, p, 'square_loss', epsilon=EPS, max_iter=400)
pseudo = T.argmax(1)                     # X_i -> its shape-matched Y_j
W0 = ridge(X_tr[ix], Y_tr[iy][pseudo])
_ = grade(W0, 'unsupervised INIT (GW, shape only)', r_sup)

In [ ]:
# 5. Step 2: CSLS self-learning refinement
def csls_refine(W, iters=20, k=10):
    npairs = 0
    for it in range(iters):
        P = l2n(X_tr @ W)
        S = P @ Y_tr.T
        # CSLS: subtract each point's mean similarity to its k nearest
        # cross-space neighbors - corrects hubness before matching
        rx = np.sort(S, 1)[:, -k:].mean(1, keepdims=True)
        ry = np.sort(S, 0)[-k:, :].mean(0, keepdims=True)
        C = 2 * S - rx - ry
        x2y, y2x = C.argmax(1), C.argmax(0)
        keep = np.where(y2x[x2y] == np.arange(len(X_tr)))[0]
        if len(keep) < 20:
            break
        W = ridge(X_tr[keep], Y_tr[x2y[keep]])
        npairs = len(keep)
        if it % 4 == 0 or it == iters - 1:
            print(f'iter {it:2d}: {npairs:4d} mutual pseudo-pairs', end='  ')
            _ = grade(W, '', r_sup)
    return W, npairs

W, npairs = csls_refine(W0)

print('\n=== FINAL SCOREBOARD ===')
r_sup2 = grade(W_sup, 'SUPERVISED ridge')
r_uns = grade(W, 'UNSUPERVISED (no pairs ever)', r_sup2)
_ = grade(W_rand, 'random floor')
chance = 1 / len(X_te)
ok = r_uns[1] >= 10 * chance and r_uns[1] >= 0.25 * r_sup2[1]
strong = r_uns[1] >= 0.50 * r_sup2[1]
print('\nVERDICT:',
      'STRONG PASS' if strong else 'PASS' if ok else 'below criteria',
      f'- unsupervised R@1 is {r_uns[1]/chance:.0f}x chance and '
      f'{100*r_uns[1]/max(r_sup2[1],1e-9):.0f}% of supervised')

## How to read the outcome

**If PASS/STRONG:** a translator between two models' spaces was
recovered **without a single paired example** — the only information
that crossed between the sides was the *shape* of each space. That is
the Platonic Representation Hypothesis made operational: the shapes are
similar enough to serve as a dictionary. It is also the security lesson
in live form: "anonymous" embeddings are translatable, hence
re-identifiable — treat embeddings as content.

**If the signature init is weak but refinement rescues it:** that is the
expected dynamic (same as unsupervised word-translation literature) —
the signature step only needs to beat noise; self-learning amplifies
any above-chance seed.

**If it fails:** also informative — this simplified method (linear map,
signature matching, mutual-NN self-learning) is far weaker than full
vec2vec (adversarial + cycle-consistency + vector-space preservation,
MLP adapters). A failure bounds what shape-only linear methods can do at
this data scale, and the honest write-up says so. The full method's
published results stand either way.

**Honesty notes for the report:** (1) this is a *simplified cousin* of
vec2vec (closest to MUSE/Artetxe self-learning), not the paper's method;
(2) both sides are COCO photos, i.e. the same input distribution — the
easiest regime for shape matching; cross-domain sides would be harder;
(3) grading uses ground-truth pairs, but training never sees them —
the split is what makes the claim checkable at all.